# Bulk Ingest: Excel → Embed → Chroma (v2 REST API)
Run the manual curl walkthrough in `manual_curl_examples.sh` FIRST so you've seen the raw request/response shapes once by hand. This notebook does the same 4 steps, batched, via Python's `requests` (same HTTP calls curl makes, just looped/batched).

In [ ]:
import pandas as pd
import requests
import sys, os
sys.path.append(os.path.abspath("../../wrapper_fix"))
sys.path.append(os.path.abspath("../../inhouse_rag_capstone"))
from inhouse_llm import API_BASE_JINA, API_KEY  # real values from your inhouse_llm.py

HOST = "localhost"
PORT = 8000
TENANT = "default_tenant"
DATABASE = "default_database"
COLLECTION_NAME = "practice_dataset"

BASE = f"http://{HOST}:{PORT}/api/v2"

## Step 1: create (if needed) and resolve the collection UUID

In [ ]:
resp = requests.post(
    f"{BASE}/tenants/{TENANT}/databases/{DATABASE}/collections",
    json={"name": COLLECTION_NAME, "configuration_json": {"hnsw": {"space": "cosine"}}},
)
print("create status:", resp.status_code, resp.text[:200])

resp = requests.get(f"{BASE}/tenants/{TENANT}/databases/{DATABASE}/collections")
collections = resp.json()
collection_id = next(c["id"] for c in collections if c["name"] == COLLECTION_NAME)
print("Collection UUID:", collection_id)

## Step 2: load the Excel dataset

In [ ]:
df = pd.read_excel("../dataset/practice_dataset.xlsx")
print(df.shape)
df.head()

## Step 3: batch-embed all rows
Adjust this function to match YOUR embedding endpoint's actual request/response shape — confirmed once already in the manual curl walkthrough. This example assumes it accepts a list of strings and returns a list of vectors.

In [ ]:
def embed_batch(texts: list[str]) -> list[list[float]]:
    """Real Jina endpoint shape, taken directly from get_embeddings() in
    inhouse_llm.py -- no more guessing at the response shape."""
    resp = requests.post(
        f"{API_BASE_JINA}/embeddings",
        headers={"Authorization": f"Bearer {API_KEY}", "Content-Type": "application/json"},
        json={"input": texts, "dimension": 1024, "embedding_type": "float"},
    )
    resp.raise_for_status()
    data = resp.json()
    return [item["embedding"] for item in data["data"]]

BATCH_SIZE = 20
all_embeddings = []
for i in range(0, len(df), BATCH_SIZE):
    batch_texts = df["text"].iloc[i:i+BATCH_SIZE].tolist()
    all_embeddings.extend(embed_batch(batch_texts))
    print(f"Embedded rows {i} to {i+len(batch_texts)}")

print("Total embeddings:", len(all_embeddings), "| dim:", len(all_embeddings[0]))

## Step 4: batch-add to Chroma

In [ ]:
def add_batch(ids, embeddings, documents, metadatas):
    resp = requests.post(
        f"{BASE}/tenants/{TENANT}/databases/{DATABASE}/collections/{collection_id}/add",
        json={"ids": ids, "embeddings": embeddings, "documents": documents, "metadatas": metadatas},
    )
    resp.raise_for_status()
    return resp

for i in range(0, len(df), BATCH_SIZE):
    chunk = df.iloc[i:i+BATCH_SIZE]
    ids = chunk["id"].tolist()
    embeddings = all_embeddings[i:i+BATCH_SIZE]
    documents = chunk["text"].tolist()
    metadatas = chunk[["topic", "type"]].to_dict("records")
    add_batch(ids, embeddings, documents, metadatas)
    print(f"Added rows {i} to {i+len(chunk)}")

## Step 5: verify the count matches

In [ ]:
resp = requests.get(f"{BASE}/tenants/{TENANT}/databases/{DATABASE}/collections/{collection_id}/count")
print("Chroma collection count:", resp.json())
print("Expected (rows in Excel):", len(df))

## Step 6: sanity queries — one per cluster, eyeball the top hit

In [ ]:
sanity_queries = {
    "cooking": "What happens to vegetables when you cook them on high heat?",
    "finance": "How does compounding affect investment growth?",
    "programming": "How does binary search reduce the number of comparisons?",
}
for topic, query_text in sanity_queries.items():
    q_vec = embed_batch([query_text])[0]
    resp = requests.post(
        f"{BASE}/tenants/{TENANT}/databases/{DATABASE}/collections/{collection_id}/query",
        json={"query_embeddings": [q_vec], "n_results": 3},
    )
    result = resp.json()
    print(f"Query ({topic}): {query_text}")
    for doc, dist in zip(result["documents"][0], result["distances"][0]):
        print(f"  [{dist:.3f}] {doc[:70]}")
    print()

### What to check
- Each query's top hit should come from the matching topic's cluster.
- The `dup_*` rows should appear as near-perfect matches whenever their parent topic is queried.
- Now open Chroma Navigator and connect to this same collection — everything you just confirmed numerically should be visible in the Visualize and Cluster tabs.